In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# 1. Define Bronze schema
bronze_schema = StructType([
    StructField("TransactionID", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("StoreID", StringType(), True),
    StructField("QuantitySold", StringType(), True),
    StructField("SalesAmount", StringType(), True),
    StructField("TransactionDate", StringType(), True)     # STRING because of corrupted row
])

# 2. File path in Databricks Volume
file_path = "/Volumes/workspace/default/muruganvolume/Retail_Sales_Data.csv"

# 3. Read data as CSV with explicit schema
df_bronze = (
    spark.read
        .format("csv")
        .option("header", "true")
        .schema(bronze_schema)
        .load(file_path)
)

df_bronze.show(5)
df_bronze.printSchema()


# Write as Bronze Delta table
df_bronze.write.format("delta").mode("overwrite").saveAsTable("default.Retail_Sales_Data")

print("Bronze table Retail_Sales_Data created successfully.")




+-------------+---------+-------+------------+-----------+---------------+
|TransactionID|ProductID|StoreID|QuantitySold|SalesAmount|TransactionDate|
+-------------+---------+-------+------------+-----------+---------------+
|            1|      188|      1|          13|     367.68|        2/16/23|
|            2|      169|      8|           7|     218.23|        10/8/23|
|            3|      178|      8|          14|      210.1|        4/19/23|
|            4|      139|      2|          11|     294.34|        4/16/23|
|            5|      122|      8|          11|     161.63|        11/1/23|
+-------------+---------+-------+------------+-----------+---------------+
only showing top 5 rows
root
 |-- TransactionID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- QuantitySold: string (nullable = true)
 |-- SalesAmount: string (nullable = true)
 |-- TransactionDate: string (nullable = true)

Bronze table Retail_Sales_Data creat

In [0]:
%sql
SELECT * FROM default.Retail_Sales_Data LIMIT 20;

TransactionID,ProductID,StoreID,QuantitySold,SalesAmount,TransactionDate
1,188,1,13,367.68,2/16/23
2,169,8,7,218.23,10/8/23
3,178,8,14,210.1,4/19/23
4,139,2,11,294.34,4/16/23
5,122,8,11,161.63,11/1/23
6,150,8,17,126.61,11/1/23
7,115,8,11,363.82,6/14/23
8,184,5,11,203.85,3/4/23
9,143,7,10,305.74,9/2/23
10,184,7,19,193.75,3/19/23


2(a) Clean Null and Corrupted Rows
• Remove records containing missing or invalid values in required fields
(TransactionID, ProductID, StoreID, QuantitySold, SalesAmount, TransactionDate)

In [0]:
from pyspark.sql.functions import col, trim, expr, try_to_date

# First, trim whitespace and rename the raw columns
clean_step1 = df_bronze.select(
    trim(col("TransactionID")).alias("TransactionID"),
    trim(col("ProductID")).alias("ProductID"),
    trim(col("StoreID")).alias("StoreID"),
    trim(col("QuantitySold")).alias("QuantitySold_raw"),
    trim(col("SalesAmount")).alias("SalesAmount_raw"),
    trim(col("TransactionDate")).alias("TransactionDate_raw")
)

def valid_required_col(c):
    return (
        col(c).isNotNull() &
        (col(c) != "") &
        (~col(c).isin("null", "NULL", "Null"))
    )

# Remove rows with missing or obviously invalid values in any required field
clean_step2 = clean_step1.filter(
    valid_required_col("TransactionID")      &
    valid_required_col("ProductID")         &
    valid_required_col("StoreID")           &
    valid_required_col("QuantitySold_raw")  &
    valid_required_col("SalesAmount_raw")   &
    valid_required_col("TransactionDate_raw")
)

display(clean_step2.limit(10))


TransactionID,ProductID,StoreID,QuantitySold_raw,SalesAmount_raw,TransactionDate_raw
1,188,1,13,367.68,2/16/23
2,169,8,7,218.23,10/8/23
3,178,8,14,210.1,4/19/23
4,139,2,11,294.34,4/16/23
5,122,8,11,161.63,11/1/23
6,150,8,17,126.61,11/1/23
7,115,8,11,363.82,6/14/23
8,184,5,11,203.85,3/4/23
9,143,7,10,305.74,9/2/23
10,184,7,19,193.75,3/19/23


In [0]:

clean_step3 = (
    clean_step2
    .withColumn("QuantitySold", col("QuantitySold_raw").cast("int"))
    .withColumn("SalesAmount", col("SalesAmount_raw").cast("double"))
    .withColumn("TransactionDate", try_to_date(col("TransactionDate_raw"), "M/d/yy"))
)

clean_step3 = clean_step3.drop(
    "QuantitySold_raw", "SalesAmount_raw", "TransactionDate_raw"
)

display(clean_step3.limit(10))
clean_step3.printSchema()

TransactionID,ProductID,StoreID,QuantitySold,SalesAmount,TransactionDate
1,188,1,13,367.68,2023-02-16
2,169,8,7,218.23,2023-10-08
3,178,8,14,210.1,2023-04-19
4,139,2,11,294.34,2023-04-16
5,122,8,11,161.63,2023-11-01
6,150,8,17,126.61,2023-11-01
7,115,8,11,363.82,2023-06-14
8,184,5,11,203.85,2023-03-04
9,143,7,10,305.74,2023-09-02
10,184,7,19,193.75,2023-03-19


root
 |-- TransactionID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- QuantitySold: integer (nullable = true)
 |-- SalesAmount: double (nullable = true)
 |-- TransactionDate: date (nullable = true)



# 2(b) Remove duplicate TransactionID entries

In [0]:

dedup_df = clean_step3.dropDuplicates(["TransactionID"])

display(dedup_df.limit(10))


TransactionID,ProductID,StoreID,QuantitySold,SalesAmount,TransactionDate
889,143,10,10,417.78,2023-09-16
50,138,9,14,191.96,2023-12-09
108,102,3,13,295.36,2023-11-11
257,107,2,19,174.27,2023-08-20
854,101,1,5,367.99,2023-11-19
577,143,4,2,167.61,2023-01-15
709,139,1,4,224.23,2023-12-12
862,115,3,2,380.93,2023-12-21
61,126,8,17,492.96,2023-07-26
244,183,8,6,58.34,2023-12-27


# 2(c) Apply business rules
QuantitySold > 0
• SalesAmount > 0
• TransactionDate must be a valid and parseable timestamp

In [0]:
from pyspark.sql.functions import col

# 2(c) Apply business rules
validated_df = (
    dedup_df
    .filter(col("QuantitySold") > 0)
    .filter(col("SalesAmount") > 0)
    .filter(col("TransactionDate").isNotNull())
)

display(validated_df.limit(10))
validated_df.printSchema()


TransactionID,ProductID,StoreID,QuantitySold,SalesAmount,TransactionDate
889,143,10,10,417.78,2023-09-16
50,138,9,14,191.96,2023-12-09
108,102,3,13,295.36,2023-11-11
257,107,2,19,174.27,2023-08-20
854,101,1,5,367.99,2023-11-19
577,143,4,2,167.61,2023-01-15
709,139,1,4,224.23,2023-12-12
862,115,3,2,380.93,2023-12-21
61,126,8,17,492.96,2023-07-26
244,183,8,6,58.34,2023-12-27


root
 |-- TransactionID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- QuantitySold: integer (nullable = true)
 |-- SalesAmount: double (nullable = true)
 |-- TransactionDate: date (nullable = true)



****2**(d) Required Analytical Transformations
Perform:
1. Standardize TransactionDate into yyyy-MM-dd.
2. Compute daily total sales per store.
3. Identify the most-sold product per store (based on total QuantitySold).
4. Identify and count stores with average daily sales > $300.**

In [0]:

#2D-1
from pyspark.sql.functions import date_format

standardized_df = validated_df.withColumn(
    "TransactionDate_std",
    date_format(col("TransactionDate"), "yyyy-MM-dd")
)


In [0]:
#2D-2
from pyspark.sql.functions import sum as _sum

df_daily_sales = (
    standardized_df
    .groupBy("StoreID", "TransactionDate")
    .agg(_sum("SalesAmount").alias("TotalDailySales"))
)

display(df_daily_sales.limit(10))



StoreID,TransactionDate,TotalDailySales
10,2023-09-16,417.78
9,2023-12-09,191.96
3,2023-11-11,295.36
2,2023-08-20,470.74
1,2023-11-19,367.99
4,2023-01-15,481.57
1,2023-12-12,224.23
3,2023-12-21,380.93
8,2023-07-26,492.96
8,2023-12-27,179.77


In [0]:
#2D-3
import pyspark.sql.functions as F

df_product_sales = (
    standardized_df
    .groupBy("StoreID", "ProductID")
    .agg(F.sum("QuantitySold").alias("TotalQuantity"))
)


In [0]:

from pyspark.sql.window import Window

w = Window.partitionBy("StoreID").orderBy(F.desc("TotalQuantity"))

df_top_products = (
    df_product_sales
    .withColumn("rank", F.rank().over(w))
    .filter(col("rank") == 1)
    .drop("rank")
)

display(df_top_products.limit(10))


StoreID,ProductID,TotalQuantity
1,159,50
10,151,58
2,109,57
3,151,50
4,167,40
5,121,49
6,174,51
7,104,39
8,136,49
9,172,30


In [0]:
#2D-4
df_avg_sales = (
    df_daily_sales
    .groupBy("StoreID")
    .agg(F.avg("TotalDailySales").alias("AvgDailySales"))
)
df_stores_gt_300 = df_avg_sales.filter(col("AvgDailySales") > 300)

display(df_stores_gt_300)

store_count = df_stores_gt_300.count()
print("Stores with average daily sales > $300:", store_count)



StoreID,AvgDailySales
10,308.86263157894746
9,312.0092647058824
3,316.86025
8,309.09409523809524
6,308.59657894736847


Stores with average daily sales > $300: 5


2(e) Store the transformed data
Write the transformed outputs into Silver Delta tables:
• Daily_Sales_Summary
• Top_Products_By_Store

In [0]:
#2E
df_daily_sales.write.format("delta").mode("overwrite") \
    .saveAsTable("default.Daily_Sales_Summary")

df_top_products.write.format("delta").mode("overwrite") \
    .saveAsTable("default.Top_Products_By_Store")




3a.Data Storage & Optimization - (a) Partitioning

In [0]:
df_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("StoreID") \
    .saveAsTable("default.Daily_Sales_Summary")

df_top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("StoreID") \
    .saveAsTable("default.Top_Products_By_Store")

#3B Z-Ordering for Optimization

In [0]:


%sql
OPTIMIZE default.Daily_Sales_Summary
ZORDER BY (TransactionDate);




path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 10, List(minCubeSize(107374182400), List(0, 0), List(10, 17380), 0, List(0, 0), 0, null), null, 0, 0, 10, 10, false, 0, 0, 1764496339821, 1764496341177, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null)"


In [0]:
%sql
OPTIMIZE default.Top_Products_By_Store
ZORDER BY (ProductID);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 10, List(minCubeSize(107374182400), List(0, 0), List(10, 8577), 0, List(0, 0), 0, null), null, 0, 0, 10, 10, false, 0, 0, 1764496419527, 1764496420582, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null)"


4. Data Analysis
Using SQL in your notebook:

a.Retrieve the total sales for the top-performing store.

b List the top 5 products across all stores by total sales.

c. Identify the highest and lowest sales dates for each store

In [0]:
#4.1 Retrieve the total sales for the top-performing store
result = spark.sql("""
SELECT
    StoreID,
    SUM(TotalDailySales) AS TotalSales
FROM default.Daily_Sales_Summary
GROUP BY StoreID
ORDER BY TotalSales DESC
LIMIT 1
""")
display(result)

StoreID,TotalSales
8,32454.879999999997


2.List the top 5 products across all stores by total sales

In [0]:
%sql
SELECT 
    ProductID,
    SUM(SalesAmount) AS TotalSales
FROM workspace.default.retail_sales_data
GROUP BY ProductID
ORDER BY TotalSales DESC
LIMIT 5;

ProductID,TotalSales
120,5395.58
114,5394.8099999999995
109,5103.839999999999
167,4386.629999999999
103,4316.700000000001


3.Identify the highest and lowest sales dates for each store

In [0]:
#Highest daily sales for each store

result = spark.sql(
    """
    SELECT StoreID, TransactionDate, TotalDailySales
    FROM (
        SELECT 
            StoreID,
            TransactionDate,
            TotalDailySales,
            ROW_NUMBER() OVER (
                PARTITION BY StoreID 
                ORDER BY TotalDailySales DESC
            ) AS rn
        FROM default.Daily_Sales_Summary
    )
    WHERE rn = 1
    """
)
display(result)

StoreID,TransactionDate,TotalDailySales
1,2023-06-03,1086.5900000000001
10,2023-09-21,861.65
2,2023-09-22,870.56
3,2023-05-12,888.5899999999999
4,2023-04-21,906.6
5,2023-12-05,854.79
6,2023-11-03,820.9300000000001
7,2023-01-19,1095.9499999999998
8,2023-03-21,1034.12
9,2023-07-30,779.47


#Lowest daily sales for each store

In [0]:
%sql
SELECT StoreID, TransactionDate, TotalDailySales
FROM (
    SELECT 
        StoreID,
        TransactionDate,
        TotalDailySales,
        ROW_NUMBER() OVER (PARTITION BY StoreID ORDER BY TotalDailySales ASC) AS rn
    FROM default.Daily_Sales_Summary
)
WHERE rn = 1;


StoreID,TransactionDate,TotalDailySales
1,2023-09-02,15.95
10,2023-06-17,14.98
2,2023-10-14,13.4
3,2023-02-07,15.77
4,2023-08-08,10.26
5,2023-11-10,11.0
6,2023-08-30,12.67
7,2023-05-18,17.67
8,2023-02-06,23.67
9,2023-10-04,26.17


5. Data Visualization



5.1. Bar Chart: Total daily sales by store

In [0]:
%sql
SELECT 
    StoreID,
    SUM(TotalDailySales) AS TotalSales
FROM default.Daily_Sales_Summary
GROUP BY StoreID
ORDER BY StoreID;



StoreID,TotalSales
1,27174.890000000003
10,29341.950000000004
2,26505.14000000001
3,25348.819999999996
4,23635.159999999996
5,26338.7
6,23453.339999999997
7,23936.839999999997
8,32454.879999999997
9,21216.63


Databricks visualization. Run in Databricks to view.

5.2. Line Graph: Sales trends over one week for store number 10.


In [0]:
%sql
SELECT 
    TransactionDate,
    TotalDailySales
FROM default.Daily_Sales_Summary
WHERE StoreID = 10
ORDER BY TransactionDate
LIMIT 7;


TransactionDate,TotalDailySales
2023-01-03,18.05
2023-01-12,244.92
2023-01-17,273.69
2023-01-19,185.05
2023-01-22,125.9
2023-01-24,596.92
2023-01-28,495.14


Databricks visualization. Run in Databricks to view.